# `mutate` — Reference
`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.
### Calling Styles
1. **Keyword arguments** (most Pythonic): `.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **Dictionary unpacking** (for new column names with spaces): `.mutate(**{"body mass kg": "body_mass_g / 1000"})`
3. **External file specification**: `.mutate("@features.txt")`
---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


species,body_mass_g,bill_length_mm,bmi
str,f64,f64,f64
"""Adelie""",3750.0,39.1,2.452888
"""Adelie""",3550.0,39.6,2.2638
"""Adelie""",3350.0,37.3,2.407837
"""Adelie""",3175.0,35.6,2.505208
"""Adelie""",3400.0,40.2,2.103908
"""Chinstrap""",3800.0,50.6,1.484166
"""Gentoo""",5550.0,50.0,2.22
"""Gentoo""",4725.0,47.4,2.103028
"""Gentoo""",5250.0,51.1,2.010562


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): division with 'String' datatypes is not allowed


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


species,body_mass_g,mass_kg,heavy
str,f64,f64,bool
"""Adelie""",4675.0,4.675,true
"""Adelie""",2850.0,2.85,false
"""Adelie""",3900.0,3.9,false
"""Adelie""",4700.0,4.7,true
"""Chinstrap""",3800.0,3.8,false
"""Chinstrap""",3600.0,3.6,false
"""Chinstrap""",4050.0,4.05,true
"""Gentoo""",5150.0,5.15,true
"""Gentoo""",5500.0,5.5,true


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


species,body_mass_g,mass_kg,mass_lb
str,f64,f64,f64
"""Adelie""",4400.0,4.4,9.700328
"""Adelie""",4150.0,4.15,9.149173
"""Adelie""",3950.0,3.95,8.708249
"""Adelie""",3900.0,3.9,8.598018
"""Adelie""",4000.0,4.0,8.81848
"""Adelie""",3650.0,3.65,8.046863
"""Gentoo""",5650.0,5.65,12.456103
"""Gentoo""",5200.0,5.2,11.464024
"""Gentoo""",4925.0,4.925,10.857753


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins
    .mutate(is_adelie="species == 'Adelie'")
    .select('species', 'is_adelie')
    .sample(10)
)


species,is_adelie
str,bool
"""Adelie""",true
"""Adelie""",true
"""Adelie""",true
"""Chinstrap""",false
"""Chinstrap""",false
"""Chinstrap""",false
"""Chinstrap""",false
"""Gentoo""",false
"""Gentoo""",false


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .mutate()
threshold = 4000

(
    penguins
    .mutate(heavy='body_mass_g >= @threshold')
    .select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


species,body_mass_g,heavy
str,f64,bool
"""Adelie""",3250.0,false
"""Adelie""",4400.0,true
"""Adelie""",3550.0,false
"""Adelie""",3400.0,false
"""Chinstrap""",3650.0,false
"""Chinstrap""",3525.0,false
"""Gentoo""",4300.0,true
"""Gentoo""",4200.0,true
"""Gentoo""",5100.0,true


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .mutate(body_mass_g='body_mass_g / 1000')
    .select('species', 'body_mass_g')
    .sample(10)
)


species,body_mass_g
str,f64
"""Adelie""",3.2
"""Adelie""",3.7
"""Adelie""",3.35
"""Adelie""",3.5
"""Adelie""",4.45
"""Adelie""",3.5
"""Chinstrap""",4.1
"""Gentoo""",5.0
"""Gentoo""",5.4


## Column names with spaces — bracket syntax
Use square brackets `[col name]` to reference column names containing spaces without shell or syntax hazards (backticks are also supported):

In [9]:
import pandas as pd
spaced = pt.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Square brackets protect column names that contain spaces
spaced.mutate(bmi="[body mass g] / [bill length mm] ** 2")


body mass g,bill length mm,bmi
i64,f64,f64
3750,39.1,2.452888
4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `filter()`.

In [10]:
(
    penguins
    .mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .filter(bmi='> 2')
    .select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


species,island,body_mass_g,bill_length_mm,bmi
str,str,f64,f64,f64
"""Adelie""","""Biscoe""",3800.0,35.9,2.948456
"""Adelie""","""Biscoe""",3450.0,35.0,2.816327
"""Adelie""","""Biscoe""",4400.0,41.3,2.579601
"""Adelie""","""Torgersen""",4700.0,42.9,2.553779
"""Adelie""","""Biscoe""",3825.0,38.1,2.635005
"""Adelie""","""Torgersen""",4000.0,40.6,2.426654
"""Adelie""","""Dream""",4475.0,37.5,3.182222
"""Gentoo""","""Biscoe""",4500.0,46.1,2.117438
"""Gentoo""","""Biscoe""",4550.0,46.5,2.10429


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


species,body_mass_g,weight_class
str,f64,str
"""Adelie""",3800.0,"""light"""
"""Adelie""",3450.0,"""light"""
"""Adelie""",3350.0,"""light"""
"""Adelie""",4700.0,"""heavy"""
"""Adelie""",3900.0,"""light"""
"""Adelie""",2900.0,"""light"""
"""Adelie""",3175.0,"""light"""
"""Gentoo""",5550.0,"""heavy"""
"""Gentoo""",4400.0,"""heavy"""


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


species,body_mass_g,size_class
str,f64,str
"""Adelie""",3900.0,"""medium"""
"""Adelie""",3300.0,"""small"""
"""Adelie""",3425.0,"""small"""
"""Adelie""",4250.0,"""medium"""
"""Chinstrap""",3525.0,"""medium"""
"""Chinstrap""",3800.0,"""medium"""
"""Chinstrap""",3500.0,"""medium"""
"""Gentoo""",5100.0,"""large"""
"""Gentoo""",5800.0,"""large"""


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .select('species', 'island', 'island_code')
    .sample(10)
)


species,island,island_code
str,str,str
"""Adelie""","""Torgersen""","""TOR"""
"""Adelie""","""Torgersen""","""TOR"""
"""Adelie""","""Dream""","""OTH"""
"""Adelie""","""Biscoe""","""BIS"""
"""Chinstrap""","""Dream""","""OTH"""
"""Gentoo""","""Biscoe""","""BIS"""
"""Gentoo""","""Biscoe""","""BIS"""
"""Gentoo""","""Biscoe""","""BIS"""
"""Gentoo""","""Biscoe""","""BIS"""


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pt.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


mobile,home,work,preferred_contact
str,str,str,str
null,"""555-5678""",null,"""555-5678"""
"""555-1234""",null,null,"""555-1234"""
null,null,"""555-9012""","""555-9012"""


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


species,bmi,mass_kg,is_heavy
str,f64,f64,bool
"""Adelie""",2.452888,3.75,false
"""Adelie""",2.435507,3.8,false
"""Adelie""",2.001121,3.25,false
"""Adelie""",null,null,null
"""Adelie""",2.561456,3.45,false


### Row-wise custom logic with Python's built-in `map()`
When custom logic requires multi-column `if/elif/else` branching or arbitrary Python functions expecting row values, wrap the scalar function in Python's built-in `map()` inside a callable `lambda d: map(func, d['a'], d['b'])`.

For dictionary value recoding, use vectorized formula `map(col, mapping)` instead.

In [16]:
def classify_penguin(species, mass_kg, island):
    if mass_kg is None or np.isnan(mass_kg):
        return "Incomplete Data"
    if species == "Gentoo" and mass_kg > 5.0:
        return "Giant Gentoo"
    elif island == "Torgersen" and mass_kg > 3.7:
        return "Prime Torgersen"
    else:
        return "Standard"

(
    penguins
    .mutate(
        mass_kg="body_mass_g / 1000",
        category=lambda d: map(classify_penguin, d["species"], d["mass_kg"], d["island"]),
    )
    .select("species", "island", "body_mass_g", "mass_kg", "category")
    .head(6)
)

species,island,body_mass_g,mass_kg,category
str,str,f64,f64,str
"""Adelie""","""Torgersen""",3750.0,3.75,"""Prime Torgersen"""
"""Adelie""","""Torgersen""",3800.0,3.8,"""Prime Torgersen"""
"""Adelie""","""Torgersen""",3250.0,3.25,"""Standard"""
"""Adelie""","""Torgersen""",null,null,"""Incomplete Data"""
"""Adelie""","""Torgersen""",3450.0,3.45,"""Standard"""
"""Adelie""","""Torgersen""",3650.0,3.65,"""Standard"""


## Loading specs from an external file — `@specs.txt`
For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from an external file. The file supports multiline expressions, `#` comments, empty lines, and column chaining (where later formulas reference columns defined earlier in the same file).
### File content of `penguin_features.txt`:
```text
# Engineering features for penguins
mass_kg = body_mass_g / 1000
# Convert kg to lbs (references mass_kg computed above)
mass_lb = mass_kg * 2.20462
```


In [17]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.mutate("@penguin_features.txt")
out = (
    result
    .select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


species,mass_kg,mass_lb
str,f64,f64
"""Adelie""",4.3,9.479866
"""Adelie""",3.5,7.71617
"""Chinstrap""",3.3,7.275246
"""Gentoo""",5.25,11.574255
"""Gentoo""",4.85,10.692407


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [18]:
spaced.mutate(ratio='[body mass g] / [bill length mm]')


body mass g,bill length mm,ratio
i64,f64,f64
3750,39.1,95.907928
4200,46.5,90.322581


## Creating new columns with spaces — string expressions `[col] = ...`

You can create columns with spaces directly using string assignment expressions with brackets `[new col] = ...` with zero dictionary unpacking.

(Dictionary unpacking `**{'col name': ...}` is also supported for kwargs).

In [19]:
# Create a new column with spaces using bracket string expression (no dict unpacking!)
(
    spaced
    .mutate("[body mass ratio] = [body mass g] / [bill length mm]")
    .select('[body mass g]', '[body mass ratio]')
    .head()
)

body mass g,body mass ratio
i64,f64
3750,95.907928
4200,90.322581


In [20]:
# Define multiple new columns with spaces at once via string expressions
(
    penguins
    .mutate(
        "[body mass kg] = body_mass_g / 1000",
        "[bill ratio] = bill_length_mm / bill_depth_mm",
    )
    .select('species', '[body mass kg]', '[bill ratio]')
    .head()
)

species,body mass kg,bill ratio
str,f64,f64
"""Adelie""",3.75,2.090909
"""Adelie""",3.8,2.270115
"""Adelie""",3.25,2.238889
"""Adelie""",null,null
"""Adelie""",3.45,1.901554


Note: When loading specs from an external file (`@specs.txt`) or via the CLI (`-mutate`), names with spaces can be written directly on the left-hand side of `=` without dictionary unpacking:

```text
# inside an external spec file or CLI flag:
my bmi = [body mass g] / [bill length mm] ** 2
```


## Grouped mutations — `df.by(...).mutate(...)`

Unlike `pt.agg()` which collapses rows ($N \to K$), `df.by(...).mutate(...)` evaluates window calculations and group-level summaries and broadcasts them back to every row ($N \to N$).

Supported features:
- **Injected aggregation functions**: `mean(x)`, `sum(x)`, `median(x)`, `min(x)`, `max(x)`, `std(x)`, `var(x)`
- **Group row count**: `n` or `n()` (dplyr / Polars convention)
- **Sequential expressions**: Later expressions within the same `mutate` call can immediately reference earlier group-derived columns (e.g. `diff = tip - avg_tip`)
- **Arbitrary output column names**: Name target columns cleanly without hardcoded prefixes
- **Multi-column grouping**: `.by('species', 'island')`
- **Spaced column names**: `[total bill]`
- **Missing group handling**: `dropna=False` retains NA groups


In [21]:
# Compute group average tip, deviation from group average, and group sample size
tips = pt.sample("tips")
(
    tips
    .select("day", "time", "total_bill", "tip")
    .by("day")
    .mutate("avg_tip = mean(tip), diff = tip - avg_tip, group_size = n")
    .head(6)
)

day,time,total_bill,tip,avg_tip,diff,group_size
cat,cat,f64,f64,f64,f64,u32
"""Sun""","""Dinner""",16.99,1.01,3.255132,-2.245132,76
"""Sun""","""Dinner""",10.34,1.66,3.255132,-1.595132,76
"""Sun""","""Dinner""",21.01,3.5,3.255132,0.244868,76
"""Sun""","""Dinner""",23.68,3.31,3.255132,0.054868,76
"""Sun""","""Dinner""",24.59,3.61,3.255132,0.354868,76
"""Sun""","""Dinner""",25.29,4.71,3.255132,1.454868,76


In [22]:
# Multi-column grouping with spaced columns
(
    tips
    .rename(columns={"total_bill": "total bill"})
    .select("day", "time", "total bill", "tip")
    .by("day", "time")
    .mutate("[avg bill] = mean([total bill]), [bill diff] = [total bill] - [avg bill], count = n()")
    .head(6)
)

day,time,total bill,tip,avg bill,bill diff,count
cat,cat,f64,f64,f64,f64,u32
"""Sun""","""Dinner""",16.99,1.01,21.41,-4.42,76
"""Sun""","""Dinner""",10.34,1.66,21.41,-11.07,76
"""Sun""","""Dinner""",21.01,3.5,21.41,-0.4,76
"""Sun""","""Dinner""",23.68,3.31,21.41,2.27,76
"""Sun""","""Dinner""",24.59,3.61,21.41,3.18,76
"""Sun""","""Dinner""",25.29,4.71,21.41,3.88,76


In [23]:
# De-meaning and z-score calculation per group
(
    penguins
    .select("species", "body_mass_g")
    .by("species")
    .mutate(
        "mean_mass = mean(body_mass_g), std_mass = std(body_mass_g), z_score = (body_mass_g - mean_mass) / std_mass"
    )
    .head(6)
)


species,body_mass_g,mean_mass,std_mass,z_score
str,f64,f64,f64,f64
"""Adelie""",3750.0,3700.662252,458.566126,0.107591
"""Adelie""",3800.0,3700.662252,458.566126,0.216627
"""Adelie""",3250.0,3700.662252,458.566126,-0.982764
"""Adelie""",null,3700.662252,458.566126,null
"""Adelie""",3450.0,3700.662252,458.566126,-0.546622
"""Adelie""",3650.0,3700.662252,458.566126,-0.11048


### Grouped vs. Overall Window Calculations (Single-Shot Grouping)

In Pytae, `.by()` establishes a single-shot, ephemeral grouping context consumed immediately by the next operation. Once `mutate()` finishes, the returned DataFrame is automatically flat and ungrouped! You can freely chain from a grouped mutation directly into an overall calculation without ever needing `ungroup()`.

In [24]:
# Comparing species-level mean to overall dataset mean in a single pipeline
(
    penguins
    .select("species", "body_mass_g")
    .by("species")
    .mutate("species_mean = mean(body_mass_g)")
    .mutate("overall_mean = mean(body_mass_g)")
    .mutate("species_vs_overall = species_mean - overall_mean")
    .head(6)
)


species,body_mass_g,species_mean,overall_mean,species_vs_overall
str,f64,f64,f64,f64
"""Adelie""",3750.0,3700.662252,4201.754386,-501.092134
"""Adelie""",3800.0,3700.662252,4201.754386,-501.092134
"""Adelie""",3250.0,3700.662252,4201.754386,-501.092134
"""Adelie""",null,3700.662252,4201.754386,-501.092134
"""Adelie""",3450.0,3700.662252,4201.754386,-501.092134
"""Adelie""",3650.0,3700.662252,4201.754386,-501.092134


### Handling missing grouping values with `dropna`

By default, `dropna=False` computes group statistics across `NA` rows as their own distinct group (no silent data loss). Pass `dropna=True` to exclude rows with `NA` in grouping columns from calculations, filling `NaN` in mutated columns while preserving total row count and index alignment:


In [25]:
sample_data = pt.DataFrame({
    "grp": ["A", "A", None, "B", None],
    "val": [10.0, 20.0, 100.0, 30.0, 200.0]
})

# dropna=False (default): NA rows form a distinct group
print("dropna=False (default):")
print(
    sample_data
    .by("grp")
    .mutate("avg = mean(val), sz = n", dropna=False)
)

# dropna=True: NA groups receive NaN
print("\ndropna=True:")
print(
    sample_data
    .by("grp")
    .mutate("avg = mean(val), sz = n", dropna=True)
)


dropna=False (default):
shape: (5, 4)
┌──────┬───────┬───────┬─────┐
│ grp  ┆ val   ┆ avg   ┆ sz  │
│ ---  ┆ ---   ┆ ---   ┆ --- │
│ str  ┆ f64   ┆ f64   ┆ u32 │
╞══════╪═══════╪═══════╪═════╡
│ A    ┆ 10.0  ┆ 15.0  ┆ 2   │
│ A    ┆ 20.0  ┆ 15.0  ┆ 2   │
│ null ┆ 100.0 ┆ 150.0 ┆ 2   │
│ B    ┆ 30.0  ┆ 30.0  ┆ 1   │
│ null ┆ 200.0 ┆ 150.0 ┆ 2   │
└──────┴───────┴───────┴─────┘

dropna=True:
shape: (5, 4)
┌──────┬───────┬──────┬──────┐
│ grp  ┆ val   ┆ avg  ┆ sz   │
│ ---  ┆ ---   ┆ ---  ┆ ---  │
│ str  ┆ f64   ┆ f64  ┆ u32  │
╞══════╪═══════╪══════╪══════╡
│ A    ┆ 10.0  ┆ 15.0 ┆ 2    │
│ A    ┆ 20.0  ┆ 15.0 ┆ 2    │
│ null ┆ 100.0 ┆ null ┆ null │
│ B    ┆ 30.0  ┆ 30.0 ┆ 1    │
│ null ┆ 200.0 ┆ null ┆ null │
└──────┴───────┴──────┴──────┘


## Comprehensive end-to-end recipe: combining all `mutate` capabilities

This master example combines all core features in a single fluent pipeline:
1. **Normal mutate**: Standard arithmetic derivation (`bill_ratio`, `mass_kg`).
2. **Sequential column referencing**: Using a freshly created column (`mass_kg`) inside subsequent expressions (`bmi`, `above_target`, `size_label`) within the *same* `mutate()` call.
3. **Binary conditional (`if_else`)**: Conditional branching into `'Heavy'` / `'Standard'`.
4. **Multi-condition tiers (`case_when`)**: Multi-tier categorization with ordered tuples and a fallback `default`.
5. **Null fallback (`coalesce`)**: Resolving the first non-null value across columns or supplying a fallback for missing data.
6. **Dictionary mapping (`map`)**: Category recoding through a dictionary lookup with fallback handling.
7. **Local variables (`@var`)**: Referencing caller-scope Python variables directly using the `@` prefix.
8. **Columns with spaces (`[col]`)**: Defining spaced columns via dict unpacking and referencing them with brackets `[mass lbs]`.
9. **Grouped window transforms (`pt.by()`)**: Computing group-level aggregations (`mean(mass_kg)`), relative offsets (`mass_diff`), and group sizes (`n`) without collapsing rows.
10. **Lambda callables**: Custom function inspecting the accumulated DataFrame with all prior mutations already attached.


In [26]:
import numpy as np
import pytae as pt

penguins = pt.sample("penguins")

# 7. Local Python variable from caller scope (referenced with @ prefix)
target_threshold = 4.2

master_df = (
    penguins
    .by("species")
    .mutate(
        # 1. Normal mutate: standard arithmetic derivation
        bill_ratio="bill_length_mm / bill_depth_mm",
        mass_kg="body_mass_g / 1000",
        
        # 3. Referencing newly created col (mass_kg) in the same mutate call
        bmi="mass_kg / ((flipper_length_mm / 1000) ** 2)",
        
        # 7a. Local variable reference (@target_threshold) from caller scope
        above_target="mass_kg > @target_threshold",
        
        # 4. if_else: binary conditional logic
        size_label="if_else(mass_kg >= 4.5, 'Heavy', 'Standard')",
        
        # 5. case_when: multi-condition tiered categorization
        weight_tier="case_when((mass_kg >= 5.0, 'XL'), (mass_kg >= 4.0, 'Large'), (mass_kg >= 3.0, 'Medium'), default='Small')",
        
        # 6. coalesce: prioritize sex, fall back to 'Undetermined' when null (row 3 has sex=NaN)
        effective_sex="coalesce(sex, 'Undetermined')",
        
        # 7b. map(): category dictionary lookup with default fallback
        species_code="map(species, {'Adelie': 'AD', 'Gentoo': 'GE', 'Chinstrap': 'CH'}, 'OTHER')",
        
        # 7c. Spaced column creation via dict unpacking & bracket referencing
        **{"mass lbs": "mass_kg * 2.20462"},
        heavy_lbs="[mass lbs] > 9.5",
        
        # 9. Grouped window transforms via standalone pt.by() without collapsing rows
        species_avg_mass="mean(mass_kg)",
        mass_diff="mass_kg - species_avg_mass",
        group_size="n",
        
        # 2. lambda: custom callable inspecting the accumulated DataFrame
        is_large=lambda df: (df["mass_kg"] > 4.0) & (df["bill_ratio"] > 2.5),
        # analysis='species', #if we don't use pt.lit, it will just assign value of speciec column
        # analysis='species2', #it will throw error because there is not 'species2 column'
        analysis="'species'", 
        
    )
)

cols_to_show = [
    "species", "species_code", "mass_kg", "species_avg_mass", "mass_diff",
    "group_size", "bmi", "size_label", "weight_tier", "effective_sex", 
    "above_target", "heavy_lbs", "is_large","analysis"
]
master_df[cols_to_show].head(8)


species,species_code,mass_kg,species_avg_mass,mass_diff,group_size,bmi,size_label,weight_tier,effective_sex,above_target,heavy_lbs,is_large,analysis
str,str,f64,f64,f64,u32,f64,str,str,str,bool,bool,bool,str
"""Adelie""","""AD""",3.75,3.700662,0.049338,152,114.46537,"""Standard""","""Medium""","""Male""",false,false,false,"""species"""
"""Adelie""","""AD""",3.8,3.700662,0.099338,152,109.839288,"""Standard""","""Medium""","""Female""",false,false,false,"""species"""
"""Adelie""","""AD""",3.25,3.700662,-0.450662,152,85.470085,"""Standard""","""Medium""","""Female""",false,false,false,"""species"""
"""Adelie""","""AD""",null,3.700662,null,152,null,"""Standard""","""Small""","""Undetermined""",null,null,false,"""species"""
"""Adelie""","""AD""",3.45,3.700662,-0.250662,152,92.619936,"""Standard""","""Medium""","""Female""",false,false,false,"""species"""
"""Adelie""","""AD""",3.65,3.700662,-0.050662,152,101.108033,"""Standard""","""Medium""","""Male""",false,false,false,"""species"""
"""Adelie""","""AD""",3.625,3.700662,-0.075662,152,110.649858,"""Standard""","""Medium""","""Female""",false,false,false,"""species"""
"""Adelie""","""AD""",4.675,3.700662,0.974338,152,122.945431,"""Heavy""","""Large""","""Male""",true,true,false,"""species"""


## Assigning Literal Constants with `quoted string literals (e.g. status="'active'")`
In `mutate()`, all string arguments are evaluated as **formulas and column expressions**, never as raw string literals:
- If you write `status="active"`, pytae searches for a column named `active`. Because no such column exists, it raises `KeyError: name 'active' is not defined`.
- Worse, if a column named `original` happens to exist, writing `source="original"` will silently copy that column instead of assigning the word `"original"`.

To assign a standalone literal constant value directly as a keyword argument without awkward nested quotes (`status="'active'"`), wrap it in `quoted string literals (e.g. status="'active'")`:
- `.mutate(status=pt.lit("active"), source=pt.lit("original"))`

> **When is `quoted string literals (e.g. status="'active'")` NOT needed?**
> 1. **Inside conditional helpers (`if_else`, `case_when`, `coalesce`)**: `quoted string literals (e.g. status="'active'")` is **completely irrelevant here**. Inside formula expressions, standard inner single quotes `'...'` already distinguish string literals from column references:
>    ```python
>    df.mutate(size="if_else(mass_kg >= 4.5, 'Heavy', 'Standard')")
>    ```
> 2. **In the CLI**: `quoted string literals (e.g. status="'active'")` is **never needed in the CLI**. In terminal commands, standard inner single quotes are always used:
>    ```bash
>    pytae data.parquet -mutate "status = 'active'"
>    ```


In [27]:
import pandas as pd
import pytae as pt

# Sample dataframe containing a column named 'original'
df = pt.DataFrame({'original': [100, 200], 'val': [1, 2]})

# 1. Unquoted string matching an existing column -> copies that column!
print("Copied column 'original':\n", df.mutate(source='original'))

# 2. With pt.lit(): assigns literal string constants cleanly and safely
print("\nWith pt.lit():\n", df.mutate(source="'original'", status="'active'"))


Copied column 'original':
 shape: (2, 3)
┌──────────┬─────┬────────┐
│ original ┆ val ┆ source │
│ ---      ┆ --- ┆ ---    │
│ i64      ┆ i64 ┆ i64    │
╞══════════╪═════╪════════╡
│ 100      ┆ 1   ┆ 100    │
│ 200      ┆ 2   ┆ 200    │
└──────────┴─────┴────────┘

With pt.lit():
 shape: (2, 4)
┌──────────┬─────┬──────────┬────────┐
│ original ┆ val ┆ source   ┆ status │
│ ---      ┆ --- ┆ ---      ┆ ---    │
│ i64      ┆ i64 ┆ str      ┆ str    │
╞══════════╪═════╪══════════╪════════╡
│ 100      ┆ 1   ┆ original ┆ active │
│ 200      ┆ 2   ┆ original ┆ active │
└──────────┴─────┴──────────┴────────┘


## Conditional Branching: `if_else` & `case_when`

Pytae provides vectorized conditional branching directly inside formula strings:
- `if_else(condition, if_true, if_false)`: Binary branching. Supports literals or picking between columns!
- `case_when((cond1, val1), (cond2, val2), default)`: Multi-tier priority branching.

In [28]:
# 1. Binary if_else returning literal categories
df_cond = penguins.mutate(size="if_else(body_mass_g > 4000, 'Heavy', 'Normal')")
df_cond.select("species", "body_mass_g", "size").head(5)

species,body_mass_g,size
str,f64,str
"""Adelie""",3750.0,"""Normal"""
"""Adelie""",3800.0,"""Normal"""
"""Adelie""",3250.0,"""Normal"""
"""Adelie""",null,"""Normal"""
"""Adelie""",3450.0,"""Normal"""


In [29]:
# 2. Dynamic column selection in if_else
# If island is 'Biscoe', select flipper_length_mm, otherwise bill_length_mm
df_col_select = penguins.mutate(metric="if_else(island == 'Biscoe', flipper_length_mm, bill_length_mm)")
df_col_select.select("species", "island", "flipper_length_mm", "bill_length_mm", "metric").head(5)

species,island,flipper_length_mm,bill_length_mm,metric
str,str,f64,f64,f64
"""Adelie""","""Torgersen""",181.0,39.1,39.1
"""Adelie""","""Torgersen""",186.0,39.5,39.5
"""Adelie""","""Torgersen""",195.0,40.3,40.3
"""Adelie""","""Torgersen""",null,null,null
"""Adelie""","""Torgersen""",193.0,36.7,36.7


In [30]:
# 3. Multi-tier branching with case_when
df_tier = penguins.mutate(
    tier="""
    case_when(
        (body_mass_g > 5000, 'Super-Heavy'),
        (body_mass_g > 3500, 'Medium'),
        'Light'
    )
    """
)
df_tier.select("species", "body_mass_g", "tier").head(6)

species,body_mass_g,tier
str,f64,str
"""Adelie""",3750.0,"""Medium"""
"""Adelie""",3800.0,"""Medium"""
"""Adelie""",3250.0,"""Light"""
"""Adelie""",null,"""Light"""
"""Adelie""",3450.0,"""Light"""
"""Adelie""",3650.0,"""Medium"""


## Referencing External Python Variables (`@var`)

Prefix local Python variables with `@` to inject them directly into formula expressions without f-strings or manual escaping.
Unquoted identifiers without `@` always resolve to DataFrame column names, preventing silent shadowing bugs.

In [31]:
cutoff = 4500.0
target_island = "Dream"

penguins.mutate(
    is_target_heavy="if_else(body_mass_g >= @cutoff and island == @target_island, 'Yes', 'No')"
).filter("is_target_heavy == 'Yes'").select("species", "island", "body_mass_g", "is_target_heavy").head(5)

species,island,body_mass_g,is_target_heavy
str,str,f64,str
"""Adelie""","""Dream""",4650.0,"""Yes"""
"""Adelie""","""Dream""",4600.0,"""Yes"""
"""Chinstrap""","""Dream""",4550.0,"""Yes"""
"""Chinstrap""","""Dream""",4800.0,"""Yes"""
"""Chinstrap""","""Dream""",4500.0,"""Yes"""


## Vectorized Value Recoding: `map(col, mapping[, fallback])`

Recode categorical values using dictionary lookups executed natively in the Polars engine with **Zero Data Loss by default**.

In [32]:
# Two-argument: recodes mapped values, preserves unmapped rows intact
penguins.mutate(
    species_code="map(species, {'Adelie': 'AD', 'Gentoo': 'GEN'})"
).select("species", "species_code").distinct().head(5)

species,species_code
str,str
"""Adelie""","""AD"""
"""Chinstrap""","""Chinstrap"""
"""Gentoo""","""GEN"""


In [33]:
# Three-argument with literal fallback: unmapped rows become 'Other'
penguins.mutate(
    species_code="map(species, {'Adelie': 'AD'}, 'Other')"
).select("species", "species_code").distinct().head(5)

species,species_code
str,str
"""Adelie""","""AD"""
"""Chinstrap""","""Other"""
"""Gentoo""","""Other"""


## Inline Type Casting: `int(_)`, `float(_)`, `str(_)`, `bool(_)`

Cast column types directly in formula expressions using standard Python constructors:

In [34]:
# Cast year to int, mass to string
df_types = penguins.mutate(
    mass_str="str(body_mass_g)"
)
print("mass_str dtype:", df_types.schema["mass_str"])

mass_str dtype: String


## Grouped Window Functions: `first`, `last`, `shift`, `cumsum`

Compute group-partitioned window metrics and baselines without collapsing table rows:

In [35]:
# Compute species mean, deviation, and first/last group baselines
penguins.by("species").mutate(
    species_mean="mean(body_mass_g)",
    mass_dev="body_mass_g - mean(body_mass_g)",
    first_mass="first(body_mass_g)",
    last_mass="last(body_mass_g)"
).select("species", "body_mass_g", "species_mean", "mass_dev", "first_mass").head(6)

species,body_mass_g,species_mean,mass_dev,first_mass
str,f64,f64,f64,f64
"""Adelie""",3750.0,3700.662252,49.337748,3750.0
"""Adelie""",3800.0,3700.662252,99.337748,3750.0
"""Adelie""",3250.0,3700.662252,-450.662252,3750.0
"""Adelie""",null,3700.662252,null,3750.0
"""Adelie""",3450.0,3700.662252,-250.662252,3750.0
"""Adelie""",3650.0,3700.662252,-50.662252,3750.0


## Dynamic Column Resolution via `col(@var)`

When writing dynamic functions or reusable pipeline steps, you can pass column names through local variables using `col(@var)` or `col('name')` rather than formatting strings:


In [36]:
target_measure = "body_mass_g"
penguins.mutate(
    normalized_measure="col(@target_measure) / 1000.0"
).select("species", "body_mass_g", "normalized_measure").head(4)


species,body_mass_g,normalized_measure
str,f64,f64
"""Adelie""",3750.0,3.75
"""Adelie""",3800.0,3.8
"""Adelie""",3250.0,3.25
"""Adelie""",null,null


## Boolean Safety in `map()` and Type Compatibility Guards

When mapping values to booleans, `map()` strictly preserves the boolean dtype (`pl.Boolean`), preventing silent string coercion into truthy `'false'` strings:


In [37]:
df_credit = pt.DataFrame({"rating": ["2B", "3C", "4D"]})
res_bool = df_credit.mutate(
    is_investment_grade="map(rating, {'2B': True, '3C': False}, default=False)"
)
print(res_bool)
print(res_bool.dtypes)


shape: (3, 2)
┌────────┬─────────────────────┐
│ rating ┆ is_investment_grade │
│ ---    ┆ ---                 │
│ str    ┆ bool                │
╞════════╪═════════════════════╡
│ 2B     ┆ true                │
│ 3C     ┆ false               │
│ 4D     ┆ false               │
└────────┴─────────────────────┘
[String, Boolean]


### Strict Type Guards Against Mixing Incompatible Types

Mixing booleans with strings or numeric types in `map()` raises an explicit `TypeError` rather than corrupting types:


In [38]:
try:
    df_credit.mutate(out="map(rating, {'2B': 'Senior'}, default=False)")
except TypeError as e:
    print(e)


map() cannot mix non-boolean mapping values with boolean default False. Polars requires homogeneous column dtypes; coercing booleans to strings would evaluate 'false' as truthy in Python.


## Unbracketed Column Names with Special Characters

Columns containing parentheses, hyphens, or special characters (common in regulatory reporting like IRBA/LGD) work seamlessly even without brackets:


In [39]:
df_irba = pt.DataFrame({
    "irba_unsecured_lgd_(1g00)": [0.25, 0.45, 0.70],
    "post_crm_lgd_(2a)": [0.20, 0.50, 0.60],
})
df_irba.mutate(
    delta="irba_unsecured_lgd_(1g00) - post_crm_lgd_(2a)",
    is_high_risk="irba_unsecured_lgd_(1g00) > 0.40",
)


irba_unsecured_lgd_(1g00),post_crm_lgd_(2a),delta,is_high_risk
f64,f64,f64,bool
0.25,0.2,0.05,false
0.45,0.5,-0.05,true
0.7,0.6,0.1,true
